## Notebook17a

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

In [ ]:
import numpy as np
import polars as pl

from funs import *
from plotnine import *
from polars import col as c
theme_set(theme_minimal())

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

### Reading the Data

In [ ]:
covid = pl.read_csv(ub + "data/it_province_covid.csv").with_columns(
    date = c.date.str.to_date()
)
covid

In [ ]:
prov = DSGeo.read_file(ub + "data/it_province.geojson")
prov

In [ ]:
it_city = pl.read_csv(ub + "data/it_cities.csv")
it_city = DSGeo.from_latlon(it_city)
it_city

**Research Questions**: How did the reported number of COVID-19 infections in Italy change over time and by region? How can we predict a spike in infections?

### Questions

1. Create a line plot of the number of COVID-19 cases in the province of Verona. Label the plot with one label per month, using only the year and month as the label.

2. Recreate the plot you made in the previous question, filtering the data to contain only the points before 1 June 2020. This is the first wave of the pandemic following local and global social distancing rules and shows the initial peak followed by slow but continual decrease in cases through the spring.

3. The daily cases are noisy. Truncate the date data at the weekly level, then group the data by week and compute the total number of cases over the week. Plot the data as you did in the first questions, using all of the data (not just the first six months).

4. Next, let's look at all of the provinces in the entire Veneto region. Repeat what you did in the previous question, but now include all provinces in Veneto. Create a line plot that shows the number of cases within each province, using color to differentiate the provinces.

5. A problem in the previous plot is that we are plotting the raw cases, but some provinces are larger than others, and what really matters is the rate of cases. To get the rate we need the population of each province. Rather than giving that directly to you (in this case it's easy to find, but in many others we would need to compute it from another dataset), we are going to calculate the population by doing the following: doing a spatial join of `it_city` into `prov`, grouping the output by province, and that summing up the population for each province. When you are confident with the output, save the result as a dataset called `prov_pop`.

6. Now, repeat what you did in question 4, but now before plotting, join the data to the `prov_pop` dataset and compute the number of cases per 100k people. Plot this rate instead of the raw cases. You should see that the rate evens out, with a single outlier (I'm not sure what's going on with this one province, but it's likely some combination of both a real spike and an issue with our population count, which only includes larger towns and cities).

7. Let's now look at the entire country at a particular moment in time. To make the plots easier to filter, replace the week truncation function you used above with the `c.date.dt.week()` function, which returns an integer of the week. For each province and week, compute the number of cases per 100k people. Then, filter the data to only include week number 12 and use `DSGeo.plot` with the `color_by` parameter set to the rate variable you created.

8. Repeat the previous question for week 13. Compare the two plots visually.

9. In this last question, I have written the code for you. Your job is just to run it and evaluate what's going on. This is code to create a moving visualization of data, something that I don't generally find particularly useful in most cases, but has a lot of utility when working with spatio-temporal data (data that has both spatial and temporal dimensions) because there is no way to show both fully on the same static plot, even with the aid of color. If for some reason you have trouble getting this to run, please look at the solutions for an itneractive version.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(10, 8))

def update(week_num):
    ax.clear()
    
    frame_data = (
        covid
        .with_columns(week = c.date.dt.week())
        .group_by(c.province, c.week)
        .agg(cases = c.cases.sum(), date = c.date.min())
        .join(prov_pop, on=c.province)
        .with_columns(rate = c.cases / c.population * 100_000)
        .filter(c.week == week_num)
        .join(prov, on=c.province)
    )

    print(frame_data['date'].first())
    
    DSGeo.plot(frame_data, color_by=c.rate, ax=ax, show=False, vmin=0, vmax=3000)
    ax.set_title(f"COVID-19 Rates - Week {week_num}, 2020")

ani = FuncAnimation(fig, update, frames=list(range(1,52)), interval=50)
HTML(ani.to_jshtml())